# 06. Embedding 시각화 — t-SNE / UMAP

**담당**: R3 (Evaluation & Reporting)  
**Stage**: S1 Prototype (W3~W4)  
**목적**: Phase 0 (Contrastive Learning) 또는 Phase 2 backbone에서 추출한 embedding을 2D로 투영하여 Grayspot Level 경계의 타당성을 시각적으로 검증한다.

---

## 이 노트북의 역할 (PRD 참조)

| 참조 섹션 | 내용 |
|-----------|------|
| PRD §3.2 | Phase 1 기준 정제: embedding 시각화 기반 Level 경계 검토 |
| PRD §5.6.4 | Representation Analysis: t-SNE/UMAP, 색상별 분포 비교, 라벨-Embedding 불일치 분석 |
| PRD §3.2.3 | 정제 완료 판단: ARI ≥ 0.6, 불일치 샘플 ≤ 10% |
| PRD §5.3.1 | 색상별 독립 모델 분리 판단: Silhouette Score ≥ 0.3 |

## 출력 산출물
- `outputs/embedding_viz/tsne_by_level.html` — Level별 색상 코딩 t-SNE (interactive)
- `outputs/embedding_viz/umap_by_level.html` — Level별 색상 코딩 UMAP (interactive)
- `outputs/embedding_viz/tsne_by_color.html` — CMYK 색상별 분포 t-SNE (interactive)
- `outputs/embedding_viz/metrics_summary.json` — ARI, Silhouette Score 등 정량 지표
- `outputs/embedding_viz/mismatch_samples.csv` — 라벨-Embedding 불일치 샘플 목록

> **S1 프로토타입 원칙**: 완벽한 모듈화보다 "돌아가는" 시각화가 우선이다.  
> 모델 checkpoint가 없는 경우 랜덤 더미 embedding으로 파이프라인을 검증할 수 있다.

---

## 0. 환경 설정 및 의존성

In [ ]:
# 필요 패키지 설치 (최초 1회)

# !pip install umap-learn scikit-learn plotly pandas numpy torch torchvision

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import LabelEncoder

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[경고] umap-learn 미설치 → UMAP 시각화 건너뜀. `pip install umap-learn` 으로 설치 가능")

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from PIL import Image
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
print(f"UMAP 사용 가능: {UMAP_AVAILABLE}")

---
## 1. 설정 (Config)

> 실제 프로젝트에서는 `config/config.yaml` 에서 로드한다. (S2 모듈화 이후)  
> S1 프로토타입 단계에서는 아래 딕셔너리를 직접 수정한다.

In [ ]:
CFG = {
    # ── 경로 설정 ──────────────────────────────────────────────────────────
    "patch_dir":      "data/patches",        # ROI 패치 이미지 디렉토리
    "label_csv":      "data/labels_v0.csv",  # 라벨 CSV (filename, yellow_level, magenta_level, cyan_level, black_level)
    "checkpoint":     None,                  # Phase 0 또는 Phase 2 checkpoint 경로 (.pt)
                                              # None 이면 랜덤 더미 embedding 생성
    "output_dir":     "outputs/embedding_viz",

    # ── 모델 설정 ──────────────────────────────────────────────────────────
    "backbone":       "efficientnet_b0",     # 'efficientnet_b0', 'resnet18', 'resnet34'
    "embedding_dim":  1280,                  # EfficientNet-B0 GAP 출력 차원 (backbone마다 다름)
    "img_size":       224,
    "batch_size":     32,

    # ── 색상 채널 설정 (PRD §6.5.2) ─────────────────────────────────────
    # 라벨 CSV의 색상별 level 컬럼 이름
    "color_columns": {
        "Y": "yellow_level",
        "M": "magenta_level",
        "C": "cyan_level",
        "K": "black_level",
    },

    # ── t-SNE / UMAP 설정 ─────────────────────────────────────────────────
    "tsne": {
        "n_components": 2,
        "perplexity":   30,
        "n_iter":       1000,
        "random_state": 42,
    },
    "umap": {
        "n_components": 2,
        "n_neighbors":  15,
        "min_dist":     0.1,
        "random_state": 42,
    },

    # ── 분석 임계값 (PRD §3.2.3, §5.3.1) ────────────────────────────────
    "ari_threshold":         0.6,   # Phase 1 정제 완료 기준
    "mismatch_threshold":    0.10,  # 불일치 샘플 비율 허용 한계
    "silhouette_threshold":  0.3,   # 색상별 독립 모델 분리 판단 기준

    # ── 시각화 색상 팔레트 ─────────────────────────────────────────────────
    "level_colors": ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#9b59b6", "#1a1a2e"],
    "cmyk_colors":  {"Y": "#f5e642", "M": "#e91e8c", "C": "#00b4d8", "K": "#444444"},
}

os.makedirs(CFG["output_dir"], exist_ok=True)
print("설정 완료:", CFG)

---
## 2. 데이터 로드

### 2.1 라벨 CSV 로드

In [ ]:
def load_labels(label_csv: str, color_columns: dict) -> pd.DataFrame:
    """
    라벨 CSV를 로드하여 long-format DataFrame으로 변환.

    원본 CSV 형식 (wide):
        filename, yellow_level, magenta_level, cyan_level, black_level, [confidence]

    반환 형식 (long):
        filename, color, level, [confidence]
        - 각 이미지당 4개 행 (Y/M/C/K)
    """
    df = pd.read_csv(label_csv)
    print(f"[CSV 로드] 원본 행 수: {len(df)}, 컬럼: {list(df.columns)}")

    # Wide → Long 변환
    records = []
    for _, row in df.iterrows():
        for color_code, col_name in color_columns.items():
            if col_name not in df.columns:
                continue
            record = {
                "filename":   row["filename"],
                "color":      color_code,
                "level":      int(row[col_name]),
                "confidence": row.get("confidence", "확실"),
            }
            records.append(record)

    long_df = pd.DataFrame(records)
    print(f"[변환 완료] Long-format 행 수: {len(long_df)}")
    print(long_df.groupby(["color", "level"]).size().unstack(fill_value=0))
    return long_df


# ── 더미 데이터 생성 (실제 CSV 없을 때 파이프라인 검증용) ─────────────────
def create_dummy_labels(n_per_class: int = 20) -> pd.DataFrame:
    """
    S1 프로토타입: 실제 라벨 CSV가 없을 때 사용하는 더미 데이터.
    Level 0~5, 색상 Y/M/C/K 조합으로 균일하게 생성.
    """
    print("[더미 데이터] 실제 CSV 미존재 → 더미 데이터 생성")
    records = []
    colors = ["Y", "M", "C", "K"]
    levels = [0, 1, 2, 3, 4, 5]
    idx = 0
    for c in colors:
        for lv in levels:
            for i in range(n_per_class):
                records.append({
                    "filename":   f"scan_dummy_{idx:04d}.png",
                    "color":      c,
                    "level":      lv,
                    "confidence": "확실" if np.random.rand() > 0.2 else "불확실",
                })
                idx += 1
    return pd.DataFrame(records)


# 실제 CSV 존재 여부에 따라 분기
if os.path.exists(CFG["label_csv"]):
    df_labels = load_labels(CFG["label_csv"], CFG["color_columns"])
else:
    df_labels = create_dummy_labels(n_per_class=20)

print(f"\n총 샘플 수: {len(df_labels)}")
df_labels.head()

### 2.2 Embedding 추출 (Backbone → GAP)

> **PRD §3.0**: Phase 0 checkpoint가 있으면 해당 backbone을 로드하고,  
> 없으면 ImageNet pretrained 또는 랜덤 초기화 backbone을 사용한다.

In [ ]:
# ── 전처리 변환 (PRD §6.5~6.9 표준 적용, Augmentation 제외) ───────────────
# S1 프로토타입에서는 단순 ImageNet 정규화 사용
# S2 모듈화 이후: preprocessing.py 의 SSOT 전처리로 교체
inference_transform = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.Grayscale(num_output_channels=3),   # 단일채널 → 3채널 복제 (PRD §6.8.4)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),  # ImageNet 기준 (S1 임시)
])


class PatchDataset(Dataset):
    """색상 패치 이미지 Dataset."""

    def __init__(self, df: pd.DataFrame, patch_dir: str, transform):
        self.df = df.reset_index(drop=True)
        self.patch_dir = Path(patch_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # 파일명에서 색상 패치 경로 구성
        # 예: data/patches/Y/scan_001.png
        color = row["color"]
        fname = row["filename"]
        img_path = self.patch_dir / color / fname

        if img_path.exists():
            img = Image.open(img_path).convert("RGB")
        else:
            # 이미지 미존재 시 더미 이미지 (S1 파이프라인 검증용)
            img = Image.fromarray(
                np.random.randint(100, 200, (CFG["img_size"], CFG["img_size"], 3),
                                  dtype=np.uint8)
            )

        return self.transform(img), idx  # tensor, 원본 인덱스 반환


def build_backbone(backbone_name: str, checkpoint: str | None, device) -> nn.Module:
    """
    Backbone 모델을 로드하고 분류기 head를 제거하여
    embedding 추출기(feature extractor)로 반환한다.

    PRD §3.0: Phase 0 checkpoint 있으면 해당 weights 로드,
              없으면 ImageNet pretrained 사용.
    """
    if backbone_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights="IMAGENET1K_V1")
        # classifier head 제거 → GAP 출력만 사용
        model.classifier = nn.Identity()
    elif backbone_name == "resnet18":
        model = models.resnet18(weights="IMAGENET1K_V1")
        model.fc = nn.Identity()
    elif backbone_name == "resnet34":
        model = models.resnet34(weights="IMAGENET1K_V1")
        model.fc = nn.Identity()
    else:
        raise ValueError(f"지원하지 않는 backbone: {backbone_name}")

    if checkpoint is not None and os.path.exists(checkpoint):
        state = torch.load(checkpoint, map_location=device)
        # Phase 0 checkpoint는 projection_head를 포함할 수 있으므로
        # backbone 관련 키만 선택적으로 로드
        backbone_state = {k.replace("backbone.", ""): v
                          for k, v in state.items() if k.startswith("backbone.")}
        if backbone_state:
            model.load_state_dict(backbone_state, strict=False)
            print(f"[체크포인트] backbone weights 로드: {checkpoint}")
        else:
            model.load_state_dict(state, strict=False)
            print(f"[체크포인트] 전체 weights 로드 시도: {checkpoint}")
    else:
        print("[체크포인트 없음] ImageNet pretrained weights 사용")

    model.eval()
    return model.to(device)


@torch.no_grad()
def extract_embeddings(model: nn.Module,
                       dataloader: DataLoader,
                       device,
                       use_dummy: bool = False,
                       n_samples: int = 0,
                       embed_dim: int = 1280) -> np.ndarray:
    """
    모델에서 embedding vector를 일괄 추출한다.
    use_dummy=True이면 랜덤 embedding을 생성 (이미지 미존재 시 파이프라인 검증용).
    """
    if use_dummy:
        print(f"[더미 Embedding] {n_samples}개 랜덤 벡터 생성 (dim={embed_dim})")
        return np.random.randn(n_samples, embed_dim).astype(np.float32)

    embeddings = []
    for batch_tensors, _ in dataloader:
        batch_tensors = batch_tensors.to(device)
        feats = model(batch_tensors)          # (B, embed_dim)
        embeddings.append(feats.cpu().numpy())

    return np.concatenate(embeddings, axis=0)


# ── 실행 ──────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

patch_dir_exists = os.path.exists(CFG["patch_dir"])

if patch_dir_exists:
    backbone = build_backbone(CFG["backbone"], CFG["checkpoint"], device)
    dataset  = PatchDataset(df_labels, CFG["patch_dir"], inference_transform)
    loader   = DataLoader(dataset, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=0, pin_memory=torch.cuda.is_available())
    embeddings = extract_embeddings(backbone, loader, device, use_dummy=False)
else:
    print(f"[경고] patch_dir 미존재: {CFG['patch_dir']} → 더미 embedding 사용")
    embeddings = extract_embeddings(
        None, None, device,
        use_dummy=True,
        n_samples=len(df_labels),
        embed_dim=CFG["embedding_dim"]
    )

print(f"\nEmbedding shape: {embeddings.shape}  (샘플 수 × embedding 차원)")

---
## 3. 차원 축소 (t-SNE / UMAP)

In [ ]:
def run_tsne(embeddings: np.ndarray, cfg: dict) -> np.ndarray:
    """t-SNE 2D 투영. 샘플 수가 많으면 perplexity 자동 조정."""
    n = len(embeddings)
    perplexity = min(cfg["perplexity"], n // 4)  # perplexity < n_samples/4 조건
    print(f"[t-SNE] n_samples={n}, perplexity={perplexity}, n_iter={cfg['n_iter']}")

    tsne = TSNE(
        n_components=cfg["n_components"],
        perplexity=perplexity,
        n_iter=cfg["n_iter"],
        random_state=cfg["random_state"],
        init="pca",
        learning_rate="auto",
        verbose=0,
    )
    proj = tsne.fit_transform(embeddings)
    print(f"[t-SNE 완료] KL Divergence: {tsne.kl_divergence_:.4f}")
    return proj


def run_umap(embeddings: np.ndarray, cfg: dict) -> np.ndarray | None:
    """UMAP 2D 투영. umap-learn 미설치 시 None 반환."""
    if not UMAP_AVAILABLE:
        return None
    print(f"[UMAP] n_neighbors={cfg['n_neighbors']}, min_dist={cfg['min_dist']}")
    reducer = umap.UMAP(
        n_components=cfg["n_components"],
        n_neighbors=cfg["n_neighbors"],
        min_dist=cfg["min_dist"],
        random_state=cfg["random_state"],
        verbose=False,
    )
    proj = reducer.fit_transform(embeddings)
    print("[UMAP 완료]")
    return proj


print("=" * 50)
print("t-SNE 실행 중... (수분 소요 가능)")
tsne_proj = run_tsne(embeddings, CFG["tsne"])

print("=" * 50)
print("UMAP 실행 중...")
umap_proj = run_umap(embeddings, CFG["umap"])

# 투영 결과를 DataFrame에 추가
df_viz = df_labels.copy()
df_viz["tsne_x"] = tsne_proj[:, 0]
df_viz["tsne_y"] = tsne_proj[:, 1]
df_viz["level_str"] = df_viz["level"].astype(str)  # plotly color 용

if umap_proj is not None:
    df_viz["umap_x"] = umap_proj[:, 0]
    df_viz["umap_y"] = umap_proj[:, 1]

print("\n차원 축소 완료.")
df_viz.head()

---
## 4. 시각화

### 4.1 t-SNE — Level별 색상 코딩

> **PRD §3.2.2 절차 2**: 기존 Level 라벨을 overlay하여 군집과의 일치도 확인

In [ ]:
def plot_projection_by_level(df: pd.DataFrame,
                              x_col: str, y_col: str,
                              method_name: str,
                              level_colors: list,
                              output_path: str | None = None) -> go.Figure:
    """
    Level별 색상 코딩 scatter plot.
    hover에 파일명, 색상, Level, 신뢰도를 표시한다.
    """
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df, x=x_col, y=y_col,
        color="level_str",
        color_discrete_map=color_map,
        symbol="color",                   # CMYK 색상을 마커 모양으로 구분
        hover_data=["filename", "color", "level", "confidence"],
        category_orders={"level_str": [str(i) for i in range(6)]},
        title=f"{method_name} — Grayspot Level별 Embedding 분포",
        labels={
            "level_str": "Level (0~5)",
            "color":     "CMYK 채널",
        },
        template="plotly_dark",
        width=900, height=650,
    )

    fig.update_traces(marker=dict(size=7, opacity=0.85,
                                   line=dict(width=0.5, color="rgba(255,255,255,0.3)")))
    fig.update_layout(
        legend_title_text="Level",
        font=dict(family="Segoe UI", size=12),
        title_font=dict(size=16),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장] {output_path}")

    return fig


fig_tsne_level = plot_projection_by_level(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    level_colors=CFG["level_colors"],
    output_path=os.path.join(CFG["output_dir"], "tsne_by_level.html"),
)
fig_tsne_level.show()

### 4.2 t-SNE — CMYK 색상별 분포

> **PRD §5.6.4**: 색상별 embedding 분포 비교 → 색상별 독립 모델 분리 필요성 판단

In [ ]:
def plot_projection_by_color(df: pd.DataFrame,
                              x_col: str, y_col: str,
                              method_name: str,
                              cmyk_colors: dict,
                              output_path: str | None = None) -> go.Figure:
    """
    CMYK 채널별 색상 코딩 scatter plot.
    Level을 마커 크기로 표현하여 Level 분포도 함께 확인 가능.
    """
    fig = px.scatter(
        df, x=x_col, y=y_col,
        color="color",
        color_discrete_map=cmyk_colors,
        size="level",                     # Level을 마커 크기로
        size_max=18,
        hover_data=["filename", "color", "level", "confidence"],
        title=f"{method_name} — CMYK 채널별 Embedding 분포 (마커 크기=Level)",
        labels={"color": "CMYK 채널"},
        template="plotly_dark",
        width=900, height=650,
    )

    fig.update_traces(marker=dict(opacity=0.75,
                                   line=dict(width=0.5, color="rgba(255,255,255,0.2)")))
    fig.update_layout(
        font=dict(family="Segoe UI", size=12),
        title_font=dict(size=16),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장] {output_path}")

    return fig


fig_tsne_color = plot_projection_by_color(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    cmyk_colors=CFG["cmyk_colors"],
    output_path=os.path.join(CFG["output_dir"], "tsne_by_color.html"),
)
fig_tsne_color.show()

### 4.3 UMAP (설치된 경우)

In [ ]:
if umap_proj is not None:
    fig_umap_level = plot_projection_by_level(
        df_viz, "umap_x", "umap_y",
        method_name="UMAP",
        level_colors=CFG["level_colors"],
        output_path=os.path.join(CFG["output_dir"], "umap_by_level.html"),
    )
    fig_umap_level.show()
else:
    print("[건너뜀] UMAP 미설치")

### 4.4 색상별 서브플롯 — Level 분포 한눈에 보기

> 4개 CMYK 채널을 2×2 서브플롯으로 비교

In [ ]:
def plot_per_color_subplot(df: pd.DataFrame,
                            x_col: str, y_col: str,
                            method_name: str,
                            level_colors: list) -> go.Figure:
    """CMYK 4채널을 2×2 서브플롯으로 Level 분포 비교."""
    colors_order = ["Y", "M", "C", "K"]
    color_titles = {"Y": "🟨 Yellow", "M": "🟥 Magenta",
                    "C": "🟦 Cyan",   "K": "⬛ Black"}

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[color_titles[c] for c in colors_order],
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )

    level_color_map = {i: level_colors[i] for i in range(6)}
    shown_in_legend = set()

    for idx, color_code in enumerate(colors_order):
        row = idx // 2 + 1
        col = idx % 2 + 1
        df_c = df[df["color"] == color_code]

        for level in range(6):
            df_lv = df_c[df_c["level"] == level]
            if len(df_lv) == 0:
                continue
            show_legend = level not in shown_in_legend
            fig.add_trace(
                go.Scatter(
                    x=df_lv[x_col], y=df_lv[y_col],
                    mode="markers",
                    name=f"Level {level}",
                    legendgroup=f"Level {level}",
                    showlegend=show_legend,
                    marker=dict(
                        color=level_color_map[level],
                        size=6, opacity=0.8,
                        line=dict(width=0.3, color="rgba(255,255,255,0.2)"),
                    ),
                    text=df_lv["filename"],
                    hovertemplate=(
                        f"색상: {color_code}<br>"
                        "Level: %{marker.color}<br>"
                        "파일: %{text}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )
            shown_in_legend.add(level)

    fig.update_layout(
        title=f"{method_name} — CMYK 채널별 Level 분포 (4분할)",
        title_font=dict(size=16),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=11),
        width=1100, height=800,
        margin=dict(l=40, r=40, t=80, b=40),
    )
    return fig


fig_subplot = plot_per_color_subplot(
    df_viz, "tsne_x", "tsne_y",
    method_name="t-SNE",
    level_colors=CFG["level_colors"],
)
fig_subplot.write_html(os.path.join(CFG["output_dir"], "tsne_per_color_subplot.html"))
print(f"[저장] {CFG['output_dir']}/tsne_per_color_subplot.html")
fig_subplot.show()

---
## 5. 정량 지표 계산

### 5.1 ARI — Embedding 군집과 Level 라벨 일치도

> **PRD §3.2.3**: ARI ≥ 0.6 이면 정제 완료 기준 충족

In [ ]:
def compute_ari(embeddings: np.ndarray,
                true_labels: np.ndarray,
                n_clusters: int = 6,
                random_state: int = 42) -> float:
    """
    Embedding에 k-means 클러스터링 적용 후 Level 라벨과의 ARI 산출.

    PRD §3.2.3:
        ARI ≥ 0.6 → Phase 1 정제 완료 기준 충족
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_labels = km.fit_predict(embeddings)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    return float(ari)


def compute_silhouette(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """
    Silhouette Score 산출.

    사용 목적:
    - labels = Level (0~5): Level 군집의 분리도
    - labels = color (Y/M/C/K): 색상별 분리도 → PRD §5.3.1 독립 모델 판단
    """
    if len(set(labels)) < 2:
        return 0.0
    # 샘플 수가 많으면 subsample (속도)
    n = len(embeddings)
    if n > 5000:
        idx = np.random.choice(n, 5000, replace=False)
        embeddings, labels = embeddings[idx], labels[idx]
    return float(silhouette_score(embeddings, labels, sample_size=min(n, 3000),
                                   random_state=42))


print("=" * 50)
print("정량 지표 계산 중...")

level_labels     = df_viz["level"].values
color_labels_enc = LabelEncoder().fit_transform(df_viz["color"].values)

# ── Level 기준 지표 ────────────────────────────────────────────────────────
ari_all = compute_ari(embeddings, level_labels, n_clusters=6)
sil_level = compute_silhouette(embeddings, level_labels)

# ── 색상별 독립 지표 (PRD §5.3.1) ─────────────────────────────────────────
sil_color = compute_silhouette(embeddings, color_labels_enc)

print(f"\n{'='*50}")
print(f"[전체] ARI (Level vs k-means)  : {ari_all:.4f}")
print(f"  → 목표: ≥ {CFG['ari_threshold']} "
      f"| {'✅ 달성' if ari_all >= CFG['ari_threshold'] else '❌ 미달 — Phase 1 추가 정제 필요'}")
print(f"[전체] Silhouette (Level)      : {sil_level:.4f}")
print(f"[전체] Silhouette (CMYK 색상)  : {sil_color:.4f}")
print(f"  → 독립 모델 분리 판단 기준: ≥ {CFG['silhouette_threshold']} "
      f"| {'⚠️ 색상별 독립 모델 검토 권장' if sil_color >= CFG['silhouette_threshold'] else '✅ 공유 모델 적합'}")

# ── 색상별 ARI ────────────────────────────────────────────────────────────
ari_by_color = {}
sil_by_color = {}
print("\n[색상별 지표]")
for color_code in ["Y", "M", "C", "K"]:
    mask = df_viz["color"].values == color_code
    emb_c = embeddings[mask]
    lv_c  = level_labels[mask]
    if len(emb_c) < 10:
        continue
    ari_c = compute_ari(emb_c, lv_c)
    sil_c = compute_silhouette(emb_c, lv_c)
    ari_by_color[color_code] = ari_c
    sil_by_color[color_code] = sil_c
    print(f"  {color_code}: ARI={ari_c:.4f}  Silhouette(Level)={sil_c:.4f}")

### 5.2 라벨-Embedding 불일치 샘플 분석

> **PRD §5.6.4**: 인간 라벨(Level)과 embedding 거리 간 불일치 샘플 추출  
> **PRD §3.2.3**: 불일치 샘플 비율 ≤ 10%

In [ ]:
def find_mismatch_samples(df: pd.DataFrame,
                           embeddings: np.ndarray,
                           n_clusters: int = 6,
                           random_state: int = 42) -> pd.DataFrame:
    """
    k-means 클러스터 할당과 Level 라벨이 불일치하는 샘플을 추출한다.

    불일치 기준:
        각 k-means 클러스터에서 가장 빈번한 Level = 대표 Level로 간주.
        해당 클러스터에서 대표 Level이 아닌 샘플 = 불일치 샘플.

    반환 컬럼:
        filename, color, level (실제), cluster_level (클러스터 대표),
        confidence, tsne_x, tsne_y
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_ids = km.fit_predict(embeddings)

    df = df.copy()
    df["cluster_id"] = cluster_ids

    # 클러스터별 대표 Level 결정
    cluster_mode = (
        df.groupby("cluster_id")["level"]
        .agg(lambda x: x.mode().iloc[0])
        .rename("cluster_level")
    )
    df = df.join(cluster_mode, on="cluster_id")

    mismatch = df[df["level"] != df["cluster_level"]].copy()
    mismatch["level_diff"] = (mismatch["level"] - mismatch["cluster_level"]).abs()
    mismatch = mismatch.sort_values("level_diff", ascending=False)

    return mismatch


df_mismatch = find_mismatch_samples(df_viz, embeddings)

mismatch_ratio = len(df_mismatch) / len(df_viz)
print(f"\n불일치 샘플 수: {len(df_mismatch)} / {len(df_viz)} "
      f"({mismatch_ratio * 100:.1f}%)")
print(f"  → 기준: ≤ {CFG['mismatch_threshold'] * 100:.0f}% "
      f"| {'✅ 기준 충족' if mismatch_ratio <= CFG['mismatch_threshold'] else '❌ 기준 초과 — 라벨 정제 필요'}")

# 색상별 불일치 현황
print("\n[색상별 불일치 현황]")
print(df_mismatch.groupby("color").size().to_string())

# 저장
mismatch_path = os.path.join(CFG["output_dir"], "mismatch_samples.csv")
df_mismatch.to_csv(mismatch_path, index=False, encoding="utf-8-sig")
print(f"\n[저장] {mismatch_path}")

df_mismatch.head(10)

### 5.3 불일치 샘플 시각화

> t-SNE 공간에서 불일치 샘플을 별도 마커(★)로 강조 표시

In [ ]:
def plot_mismatch_overlay(df_all: pd.DataFrame,
                           df_mismatch: pd.DataFrame,
                           x_col: str, y_col: str,
                           level_colors: list,
                           output_path: str | None = None) -> go.Figure:
    """불일치 샘플을 ★ 마커로 overlay한 t-SNE scatter plot."""
    color_map = {str(i): c for i, c in enumerate(level_colors)}

    fig = px.scatter(
        df_all, x=x_col, y=y_col,
        color="level_str",
        color_discrete_map=color_map,
        title="t-SNE — 불일치 샘플 강조 (★)",
        template="plotly_dark",
        hover_data=["filename", "color", "level"],
        width=900, height=650,
    )
    fig.update_traces(marker=dict(size=6, opacity=0.5))

    # 불일치 샘플 overlay
    if len(df_mismatch) > 0 and x_col in df_mismatch.columns:
        fig.add_trace(go.Scatter(
            x=df_mismatch[x_col],
            y=df_mismatch[y_col],
            mode="markers",
            name="불일치 샘플 (★)",
            marker=dict(
                symbol="star",
                size=14,
                color="#ff7aa2",
                line=dict(width=1, color="white"),
                opacity=0.95,
            ),
            text=df_mismatch.apply(
                lambda r: f"{r['filename']}<br>실제:{r['level']} 클러스터:{r['cluster_level']}",
                axis=1
            ),
            hovertemplate="%{text}<extra></extra>",
        ))

    fig.update_layout(
        font=dict(family="Segoe UI", size=12),
        margin=dict(l=40, r=40, t=60, b=40),
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장] {output_path}")
    return fig


# df_mismatch에 tsne 좌표 병합 (인덱스 기준)
df_mismatch_with_coords = df_mismatch.copy()

fig_mismatch = plot_mismatch_overlay(
    df_viz, df_mismatch_with_coords,
    "tsne_x", "tsne_y",
    level_colors=CFG["level_colors"],
    output_path=os.path.join(CFG["output_dir"], "tsne_mismatch_overlay.html"),
)
fig_mismatch.show()

---
## 6. 지표 요약 및 Phase 1 판단

> **PRD §3.2.3 정제 완료 판단 기준** 자동 평가

In [ ]:
# ── 지표 집계 ─────────────────────────────────────────────────────────────
metrics_summary = {
    "meta": {
        "n_samples":       int(len(df_viz)),
        "backbone":        CFG["backbone"],
        "checkpoint":      CFG["checkpoint"],
        "tsne_kl":         None,   # tsne.kl_divergence_ (저장하려면 위에서 캡처 필요)
    },
    "global": {
        "ARI":                     round(ari_all, 4),
        "ARI_threshold":            CFG["ari_threshold"],
        "ARI_pass":                 ari_all >= CFG["ari_threshold"],
        "silhouette_level":         round(sil_level, 4),
        "silhouette_color":         round(sil_color, 4),
        "silhouette_color_threshold": CFG["silhouette_threshold"],
        "independent_model_recommended": sil_color >= CFG["silhouette_threshold"],
        "mismatch_count":           int(len(df_mismatch)),
        "mismatch_ratio":           round(mismatch_ratio, 4),
        "mismatch_threshold":       CFG["mismatch_threshold"],
        "mismatch_pass":            mismatch_ratio <= CFG["mismatch_threshold"],
    },
    "by_color": {
        color: {
            "ARI":        round(ari_by_color.get(color, -1), 4),
            "silhouette": round(sil_by_color.get(color, -1), 4),
        }
        for color in ["Y", "M", "C", "K"]
    },
}

# 저장
metrics_path = os.path.join(CFG["output_dir"], "metrics_summary.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2)
print(f"[저장] {metrics_path}")

# ── Phase 1 판단 출력 ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Phase 1 정제 완료 판단 (PRD §3.2.3)")
print("=" * 60)

cond_ari      = metrics_summary["global"]["ARI_pass"]
cond_mismatch = metrics_summary["global"]["mismatch_pass"]

print(f"  [1] ARI ≥ {CFG['ari_threshold']}         : "
      f"{ari_all:.4f}  →  {'✅ 충족' if cond_ari else '❌ 미달'}")
print(f"  [2] 불일치 비율 ≤ {CFG['mismatch_threshold']*100:.0f}%  : "
      f"{mismatch_ratio*100:.1f}%  →  {'✅ 충족' if cond_mismatch else '❌ 미달'}")
print()

if cond_ari and cond_mismatch:
    print("  🎉 판정: Phase 1 정제 완료 기준 충족 → Phase 2 진입 가능")
else:
    print("  ⚠️  판정: 기준 미달 → 불일치 샘플 검토 후 라벨 재정제 필요")
    print("           mismatch_samples.csv 확인 후 Phase 1 재수행")

print()
independent = metrics_summary["global"]["independent_model_recommended"]
print(f"  [3] 색상별 독립 모델 검토 (PRD §5.3.1)")
print(f"      Silhouette(CMYK) = {sil_color:.4f}  "
      f"{'≥' if independent else '<'} {CFG['silhouette_threshold']}")
print(f"      → {'⚠️ 색상별 독립 모델 분리 검토 권장' if independent else '✅ 공유 모델 적합'}")

---
## 7. 지표 시각화 대시보드

In [ ]:
def plot_metrics_dashboard(metrics: dict,
                            output_path: str | None = None) -> go.Figure:
    """ARI, Silhouette, 불일치 비율을 Gauge + Bar로 시각화."""
    g = metrics["global"]
    colors_order = ["Y", "M", "C", "K"]

    fig = make_subplots(
        rows=2, cols=3,
        specs=[
            [{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
            [{"type": "bar",        "colspan": 3},     None,                    None],
        ],
        subplot_titles=[
            "ARI (Level vs k-means)",
            "불일치 샘플 비율",
            "Silhouette (CMYK 색상)",
            "색상별 ARI",
        ],
        vertical_spacing=0.15,
    )

    # Gauge: ARI
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=g["ARI"],
        number={"font": {"size": 32}},
        gauge={
            "axis": {"range": [0, 1]},
            "bar": {"color": "#50e3c2"},
            "threshold": {"line": {"color": "#ff7aa2", "width": 3},
                           "value": g["ARI_threshold"]},
            "bgcolor": "#0b1220",
        },
        title={"text": f"목표 ≥ {g['ARI_threshold']}"},
    ), row=1, col=1)

    # Gauge: 불일치 비율 (낮을수록 좋음)
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=round(g["mismatch_ratio"] * 100, 1),
        number={"suffix": "%", "font": {"size": 32}},
        gauge={
            "axis": {"range": [0, 50]},
            "bar": {"color": "#66d9ff"},
            "threshold": {"line": {"color": "#ff7aa2", "width": 3},
                           "value": g["mismatch_threshold"] * 100},
            "bgcolor": "#0b1220",
        },
        title={"text": f"목표 ≤ {g['mismatch_threshold']*100:.0f}%"},
    ), row=1, col=2)

    # Gauge: Silhouette (색상)
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=g["silhouette_color"],
        number={"font": {"size": 32}},
        gauge={
            "axis": {"range": [-1, 1]},
            "bar": {"color": "#c792ea"},
            "threshold": {"line": {"color": "#ffb347", "width": 3},
                           "value": g["silhouette_color_threshold"]},
            "bgcolor": "#0b1220",
        },
        title={"text": f"독립 모델 기준 ≥ {g['silhouette_color_threshold']}"},
    ), row=1, col=3)

    # Bar: 색상별 ARI
    by_color = metrics["by_color"]
    bar_colors = ["#f5e642", "#e91e8c", "#00b4d8", "#aaaaaa"]
    fig.add_trace(go.Bar(
        x=colors_order,
        y=[by_color[c]["ARI"] for c in colors_order],
        marker_color=bar_colors,
        text=[f"{by_color[c]['ARI']:.3f}" for c in colors_order],
        textposition="outside",
        name="ARI",
    ), row=2, col=1)

    # 목표선
    fig.add_hline(y=g["ARI_threshold"], line_dash="dot",
                   line_color="#ff7aa2",
                   annotation_text=f"목표 {g['ARI_threshold']}",
                   row=2, col=1)

    fig.update_layout(
        title="Embedding 품질 지표 대시보드",
        title_font=dict(size=18),
        template="plotly_dark",
        font=dict(family="Segoe UI", size=12),
        width=1100, height=700,
        margin=dict(l=40, r=40, t=80, b=40),
        showlegend=False,
    )

    if output_path:
        fig.write_html(output_path)
        print(f"[저장] {output_path}")
    return fig


fig_dashboard = plot_metrics_dashboard(
    metrics_summary,
    output_path=os.path.join(CFG["output_dir"], "metrics_dashboard.html"),
)
fig_dashboard.show()

---
## 8. 산출물 요약

In [ ]:
print("=" * 60)
print("생성된 산출물 목록")
print("=" * 60)

output_files = [
    ("tsne_by_level.html",           "t-SNE — Level별 색상 코딩 (interactive)"),
    ("tsne_by_color.html",           "t-SNE — CMYK 채널별 분포 (interactive)"),
    ("tsne_per_color_subplot.html",  "t-SNE — CMYK 4분할 서브플롯"),
    ("tsne_mismatch_overlay.html",   "t-SNE — 불일치 샘플 강조 (★)"),
    ("umap_by_level.html",           "UMAP — Level별 색상 코딩 (umap-learn 필요)"),
    ("metrics_dashboard.html",       "Embedding 품질 지표 대시보드"),
    ("metrics_summary.json",         "ARI / Silhouette / 불일치 비율 수치"),
    ("mismatch_samples.csv",         "라벨-Embedding 불일치 샘플 목록"),
]

for fname, desc in output_files:
    full_path = os.path.join(CFG["output_dir"], fname)
    exists = "✅" if os.path.exists(full_path) else "⬜"
    print(f"  {exists}  {fname:<40} {desc}")

print()
print("다음 단계 (PRD §3.2.2):")
print("  1. tsne_by_level.html 을 열고 Level 군집 경계를 육안으로 확인")
print("  2. mismatch_samples.csv 의 불일치 샘플을 순서대로 검토하여 라벨 수정")
print("  3. metrics_dashboard 에서 ARI / 불일치 비율 기준 충족 여부 확인")
print("  4. 기준 미달 시 → 라벨 수정 후 이 노트북 재실행 (Phase 1 루프)")
print("  5. 기준 충족 시 → 정제된 CSV를 notebooks/03_training.ipynb 에 전달")